# NLP Exercise 5: BERTs with Masked Language Model
---

In [28]:
%pip install datasets transformers torch tqdm scikit-learn evaluate rouge_score
%pip install transformers[torch]

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


# Import modules

In [1]:
from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import pipeline, get_scheduler, default_data_collator
from transformers import AutoTokenizer, DataCollatorForLanguageModeling, AutoModelForMaskedLM
from transformers import DataCollatorForSeq2Seq, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm
import math
import evaluate
import numpy as np

c:\Users\minhn\anaconda3\envs\general\Lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.2)
  from scipy.sparse import csr_matrix, issparse
c:\Users\minhn\anaconda3\envs\general\Lib\site-packages\torch\xpu\__init__.py:61: UserWarning: XPU device count is zero! (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10\xpu\XPUFunctions.cpp:115.)
  return torch._C._xpu_getDeviceCount()


# Read, clean and split text to list of fables

In [2]:
text_filename = 'pg11339.txt'
# Load documents
with open(text_filename, 'r', encoding='utf-8') as file:
    story_text = file.read()

# Get only the story titles and content
chapter_start = "THE FOX AND THE GRAPES\n\n\nA hungry Fox"
chapter_end = "ILLUSTRATIONS\n\n\n[Illustration: THE HARE AND THE TORTOISE]"
start_idx = story_text.find(chapter_start)
end_idx = story_text.find(chapter_end)
text = story_text[start_idx:end_idx].strip()
fables = text.split('\n' * 5)

# 1) Masked Language Model

## Load fables to dataset

In [3]:
fables_train, fables_test = train_test_split(fables, train_size=0.9, test_size=0.1)
fables_dataset = DatasetDict({
    'train': Dataset.from_dict({'text': fables_train}),
    'test': Dataset.from_dict({'text': fables_test}),
}) 
fables_dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 255
    })
    test: Dataset({
        features: ['text'],
        num_rows: 29
    })
})

## Tokenize fables

In [4]:
# Define tokenizer
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

# Tokenize function to tokenize the whole dataset
def tokenize(data):
    tokenized_data = tokenizer(data["text"])
    return tokenized_data

# Map function
tokenized_data = fables_dataset.map(tokenize, batched=True, remove_columns=["text"])

Map:   0%|          | 0/255 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (660 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/29 [00:00<?, ? examples/s]

In [5]:
tokenized_data

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 255
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 29
    })
})

## Preprocessing for training loop

In [6]:
def group_texts(data):
    chunk_size = 128
    # concatenate texts
    concatenated_sequences = {k: sum(data[k], []) for k in data.keys()}
    # calculate the total number of tokens after concatenation
    total_concat_length = len(concatenated_sequences[list(data.keys())[0]])
    # drop the last chunk if is smaller than the chunk size
    total_length = (total_concat_length // chunk_size) * chunk_size

    # split the concatenated sentences into chunks using the total length
    result = {k: [t[i: i + chunk_size] for i in range(0, total_length, chunk_size)]
    for k, t in concatenated_sequences.items()}

    '''we create a new labels column which is a copy of the input_ids of the processed text data,
    we need to predict randomly masked tokens in the input batch and the labels column serve as
    ground truth for our masked language model to learn from. '''

    result["labels"] = result["input_ids"].copy()

    return result

processed_dataset = tokenized_data.map(group_texts, batched = True)


Map:   0%|          | 0/255 [00:00<?, ? examples/s]

Map:   0%|          | 0/29 [00:00<?, ? examples/s]

In [7]:
data_collator = DataCollatorForLanguageModeling(tokenizer = tokenizer, mlm_probability = 0.15)

# we shall insert mask randomly in the sentence
def insert_random_mask(batch):
    features = [dict(zip(batch, t)) for t in zip(*batch.values())]
    masked_inputs = data_collator(features)
    # Create a new "masked" column for each column in the dataset
    return {"masked_" + k: v.numpy() for k, v in masked_inputs.items()}

In [8]:
eval_dataset = processed_dataset["test"].map(
    insert_random_mask,
    batched=True,
    remove_columns=processed_dataset["test"].column_names,
)
eval_dataset = eval_dataset.rename_columns(
    {
        "masked_input_ids": "input_ids",
        "masked_attention_mask": "attention_mask",
        "masked_labels": "labels",
    }
)

Map:   0%|          | 0/37 [00:00<?, ? examples/s]

In [9]:
eval_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 37
})

## Define DataLoader and relevant model params

In [10]:
batch_size = 16

# load the train dataset for traing
train_dataloader = DataLoader(processed_dataset["train"], shuffle=True, batch_size=batch_size, collate_fn=data_collator)
# load the test dataset for evaluation
eval_dataloader = DataLoader(eval_dataset, batch_size=batch_size, collate_fn=default_data_collator)

# initialize pretrained bert model
model = AutoModelForMaskedLM.from_pretrained('distilbert-base-uncased')

# set the optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)

## Training loop

In [11]:
# Define the number of epochs
epochs = 10

# Define the learning rate scheduler
num_training_steps = epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    name="linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

# Move model to GPU if available
if torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


# Training and evaluation loop
progress_bar = tqdm(range(num_training_steps))

for epoch in range(epochs):
    # Training
    model.train()
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    # Evaluation
    model.eval()
    eval_loss = 0
    for batch in eval_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        eval_loss += outputs.loss.item()

    eval_loss = eval_loss / len(eval_dataloader)
    perplexity = math.exp(eval_loss)
    print(f"Epoch {epoch + 1}/{epochs}, Training Loss: {loss.item()}, Evaluation Loss: {eval_loss}, Perplexity: {perplexity}")

  0%|          | 0/210 [00:00<?, ?it/s]

Epoch 1/10, Training Loss: 2.3898091316223145, Evaluation Loss: 2.097883661588033, Perplexity: 8.148905808290216
Epoch 2/10, Training Loss: 2.2687995433807373, Evaluation Loss: 2.0357099771499634, Perplexity: 7.657686978716001
Epoch 3/10, Training Loss: 2.403803825378418, Evaluation Loss: 1.9933788379033406, Perplexity: 7.340293571203258
Epoch 4/10, Training Loss: 1.9762531518936157, Evaluation Loss: 1.9537267287572224, Perplexity: 7.054930462399701
Epoch 5/10, Training Loss: 2.2496140003204346, Evaluation Loss: 1.942133863766988, Perplexity: 6.97361585137176
Epoch 6/10, Training Loss: 2.1527323722839355, Evaluation Loss: 1.9297457536061604, Perplexity: 6.887758831101384
Epoch 7/10, Training Loss: 2.0050406455993652, Evaluation Loss: 1.9315982262293498, Perplexity: 6.90053004127754
Epoch 8/10, Training Loss: 2.1982882022857666, Evaluation Loss: 1.9067993958791096, Perplexity: 6.731509386692579
Epoch 9/10, Training Loss: 1.8355849981307983, Evaluation Loss: 1.9110360145568848, Perplexit

## Save and test models

In [12]:
model.save_pretrained("BERT_exercise_masked_model")
tokenizer.save_pretrained("BERT_exercise_masked_model")

('BERT_exercise_masked_model\\tokenizer_config.json',
 'BERT_exercise_masked_model\\special_tokens_map.json',
 'BERT_exercise_masked_model\\vocab.txt',
 'BERT_exercise_masked_model\\added_tokens.json',
 'BERT_exercise_masked_model\\tokenizer.json')

In [13]:
# Example sentences with masks
fable_sentence_1 = "venus was very <mask> about it, and changed her at once into a beautiful maiden"
fable_sentence_2 = "they decided to <mask> it in order to secure the whole store of precious metal at once"
sentence_1 = "he was aware of his <mask> yesterday morning"
sentence_2 = "it is really difficult to <mask> treat everyone everyday"

In [14]:
# Default masked model
default_pred_model = pipeline("fill-mask")

preds_1 = default_pred_model(fable_sentence_1)
preds_2 = default_pred_model(fable_sentence_2)
preds_3 = default_pred_model(sentence_1)
preds_4 = default_pred_model(sentence_2)

print('\nFable sentence 1:')
for i, pred in enumerate(preds_1):
    print(f"{i + 1}. {pred['sequence']}")
print('\nFable sentence 2:')
for i, pred in enumerate(preds_2):
    print(f"{i + 1}. {pred['sequence']}")
print('\nSentence 1:')
for i, pred in enumerate(preds_3):
    print(f"{i + 1}. {pred['sequence']}")
print('\nSentence 2:')
for i, pred in enumerate(preds_4):
    print(f"{i + 1}. {pred['sequence']}")

No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8 (https://huggingface.co/distilbert/distilroberta-base).
Using a pipeline without specifying a model name and revision in production is not recommended.
Some weights of the model checkpoint at distilbert/distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu



Fable sentence 1:
1. venus was very happy about it, and changed her at once into a beautiful maiden
2. venus was very gracious about it, and changed her at once into a beautiful maiden
3. venus was very passionate about it, and changed her at once into a beautiful maiden
4. venus was very enthusiastic about it, and changed her at once into a beautiful maiden
5. venus was very clever about it, and changed her at once into a beautiful maiden

Fable sentence 2:
1. they decided to hoard it in order to secure the whole store of precious metal at once
2. they decided to use it in order to secure the whole store of precious metal at once
3. they decided to scrap it in order to secure the whole store of precious metal at once
4. they decided to destroy it in order to secure the whole store of precious metal at once
5. they decided to encrypt it in order to secure the whole store of precious metal at once

Sentence 1:
1. he was aware of his whereabouts yesterday morning
2. he was aware of his 

In [16]:
# My own masked model
pred_model = pipeline("fill-mask", model="BERT_exercise_masked_model", tokenizer="BERT_exercise_masked_model")

preds_1 = pred_model(fable_sentence_1.replace('<mask>', '[MASK]'))
preds_2 = pred_model(fable_sentence_2.replace('<mask>', '[MASK]'))
preds_3 = pred_model(sentence_1.replace('<mask>', '[MASK]'))
preds_4 = pred_model(sentence_2.replace('<mask>', '[MASK]'))

print('\nFable sentence 1:')
for i, pred in enumerate(preds_1):
    print(f"{i + 1}. {pred['sequence']}")
print('\nFable sentence 2:')
for i, pred in enumerate(preds_2):
    print(f"{i + 1}. {pred['sequence']}")
print('\nSentence 1:')
for i, pred in enumerate(preds_3):
    print(f"{i + 1}. {pred['sequence']}")
print('\nSentence 2:')
for i, pred in enumerate(preds_4):
    print(f"{i + 1}. {pred['sequence']}")

Device set to use cpu



Fable sentence 1:
1. venus was very unhappy about it, and changed her at once into a beautiful maiden
2. venus was very pleased about it, and changed her at once into a beautiful maiden
3. venus was very proud about it, and changed her at once into a beautiful maiden
4. venus was very happy about it, and changed her at once into a beautiful maiden
5. venus was very excited about it, and changed her at once into a beautiful maiden

Fable sentence 2:
1. they decided to destroy it in order to secure the whole store of precious metal at once
2. they decided to sell it in order to secure the whole store of precious metal at once
3. they decided to open it in order to secure the whole store of precious metal at once
4. they decided to close it in order to secure the whole store of precious metal at once
5. they decided to secure it in order to secure the whole store of precious metal at once

Sentence 1:
1. he was aware of his presence yesterday morning
2. he was aware of his absence yester

# 2) Text summarization

In [3]:
pretrained_summarization_model = 'facebook/bart-large-cnn'

## Load dataset

### Option 1: Load fable summary dataset

In [4]:
summary_data = load_dataset('csv', data_files={'train': 'fable-summary-train.csv', 'test': 'fable-summary-test.csv'}, delimiter='|')
print(summary_data)

DatasetDict({
    train: Dataset({
        features: ['text', 'summary'],
        num_rows: 45
    })
    test: Dataset({
        features: ['text', 'summary'],
        num_rows: 7
    })
})


### Option 2: Load booksum dataset

In [4]:
summary_data = load_dataset('kmfoda/booksum')
# Keep only 'chapter' (original text) and 'summary_text' columns
summary_data = summary_data.remove_columns(['bid', 'is_aggregate', 'source', 'chapter_path', 'summary_path', 'book_id', 'summary_id', 'content', 'summary', 'chapter_length', 'summary_name', 'summary_url', 'summary_analysis', 'summary_length', 'analysis_length'])
summary_data = summary_data.rename_columns({'chapter': 'text', 'summary_text': 'summary'}) # Rename the remaining two columns
print(summary_data)

DatasetDict({
    train: Dataset({
        features: ['text', 'summary'],
        num_rows: 9600
    })
    validation: Dataset({
        features: ['text', 'summary'],
        num_rows: 1484
    })
    test: Dataset({
        features: ['text', 'summary'],
        num_rows: 1431
    })
})


## Tokenize data

In [5]:
# Define tokenizer
summary_tokenizer = AutoTokenizer.from_pretrained(pretrained_summarization_model)

# Tokenize function to tokenize the whole dataset
prefix = "summarize: "
def preprocess_function(examples):
    inputs = [prefix + doc for doc in examples["text"]]
    model_inputs = summary_tokenizer(inputs, max_length=1024, truncation=True)

    labels = summary_tokenizer(text_target=examples["summary"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

summary_tokenized = summary_data.map(preprocess_function, batched=True)

config.json: 0.00B [00:00, ?B/s]

c:\Users\minhn\anaconda3\envs\general\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\minhn\.cache\huggingface\hub\models--facebook--bart-large-cnn. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/45 [00:00<?, ? examples/s]

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

## Define data collator and evaluator function

In [6]:
data_collator = DataCollatorForSeq2Seq(tokenizer=summary_tokenizer, model=pretrained_summarization_model)
rouge = evaluate.load("rouge")
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = summary_tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, summary_tokenizer.pad_token_id)
    decoded_labels = summary_tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    prediction_lens = [np.count_nonzero(pred != summary_tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

## Define training parameters and train the model

In [ ]:
# Use a small subset of the dataset for a quick training (if you choose option 2 above)
small_summary_tokenized = summary_tokenized["train"].train_test_split(train_size=50, test_size=10, seed=42)

In [23]:
model = AutoModelForSeq2SeqLM.from_pretrained(pretrained_summarization_model)

# Move model to GPU if available
if torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

training_args = Seq2SeqTrainingArguments(
    output_dir="BART_summarization_model",
    eval_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=8,
    weight_decay=0.001,
    save_total_limit=3,
    num_train_epochs=4,
    predict_with_generate=True,
    fp16=True, #change to bf16=True for XPU
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=summary_tokenized["train"], # small_summary_tokenized["train"] (for option 2)
    eval_dataset=summary_tokenized["test"], # small_summary_tokenized["test"] (for option 2)
    processing_class=summary_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [24]:
trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,No log,1.811258,0.474200,0.158500,0.307000,0.305900,93.142900
2,No log,1.764282,0.457400,0.166200,0.316200,0.315900,86.000000
3,No log,1.747294,0.496800,0.203100,0.343300,0.343600,83.714300
4,No log,1.741280,0.485500,0.187500,0.343600,0.342400,79.571400


c:\Users\minhn\anaconda3\envs\general\Lib\site-packages\transformers\modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
c:\Users\minhn\anaconda3\envs\general\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=12, training_loss=1.7323490778605144, metrics={'train_runtime': 495.7894, 'train_samples_per_second': 0.363, 'train_steps_per_second': 0.024, 'total_flos': 151593622806528.0, 'train_loss': 1.7323490778605144, 'epoch': 4.0})

## Test the models

In [47]:
# Default summarization model
default_summary_model = pipeline("summarization")

text = fables[-3] # THE GOATHERD AND THE WILD GOATS

result = default_summary_model(text)

print(result[0]['summary_text'])

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


 A Goatherd was tending his goats out at pasture when he saw a number of Wild Goats approach and mingle with his flock . He only gave his own goats enough food to keep them from starving, but he gave the wild Goats as much as they could eat and more . When the weather improved, he took them out to pasture again; but no sooner had they got near the hills they broke away from the flock and scampered off .


In [30]:
# My own summarization model
# After training, change the checkpoint number if it is different
summary_model = pipeline("summarization", model="BART_summarization_model/checkpoint-12")

text = fables[-3] # THE GOATHERD AND THE WILD GOATS

result = summary_model(text)

print(result[0]['summary_text'])

Device set to use cpu


The Goatherd feeds the Wild Goats more than his own goats because he is anxious for them to stay with him. The Wild goats tell him that if he treats them so well they will not want to leave him. When the weather improves, the Wild goats scamper away from the flock and scamper off.
